# Example 4: BRAM content injection and ECC recovery

This example injects upsets into the content of one BRAM. Then it shows which words read back wrong, and which of them an error-correcting code (ECC) in the design repairs:

- One flipped bit in a byte: the ECC corrects it.
- Two or more flipped bits in the same byte: the ECC cannot correct them, and it can also change a correct bit.

## Design

![Block diagram of brams_top. A BRAM in a pblock has an encoder on the write path. On the read path, a falling-edge hold register on the gated clock feeds a decoder and a multiplexer. USER1 controls the clock gate, USER2 selects the address, USER3 carries the data and USER4 enables the ECC correction.](images/example4_brams_top_abstract.svg)

The design [`brams_top`](../hw/sources_1/new/brams_top.vhd) has one BRAM (a RAMB36), `soft_sec_bram`, in the pblock `pblock_soft_sec_bram`. The BRAM has 1024 words of 36 bits:

| Bits of a word | Contents |
| --- | --- |
| 23:0 | 3 data bytes |
| 35:24 | 12 check bits: 4 for each data byte |

The ECC is a Hamming SEC (single error correction) code for each byte, in the fabric around the BRAM:

- **Write:** an encoder calculates the 4 check bits of each data byte. The BRAM stores the data bytes and their check bits.
- **Read:** a decoder checks each byte with its 4 check bits. If one of these 12 bits is flipped, the decoder corrects it.
- **USER4** selects what the script reads: the raw data (correction off) or the corrected data (correction on).

The ECC is in the fabric because the built-in ECC of the BRAM does not see an SK-RAFT injection. [Example 6](../example6/README.md) explains why.

The script controls the design through four BSCAN registers:

| Register | Width | Access | Contents |
| --- | --- | --- | --- |
| USER1 | 1 bit | write | The clock enable of the BRAM. The script stops the clock during each injection. |
| USER2 | 16 bits | write | The address. Only the low 10 bits have an effect. |
| USER3 | 24 bits | write and read | The data at the address. A write of USER3 writes the word. A read of USER3 reads the word, and does not write it (refer to [`auxiliary_components/bscan`](../../auxiliary_components/bscan/README.md)). |
| USER4 | 1 bit | write | The ECC correction: 1 = on, 0 = off. It applies to the read path only. The reset value is 0. |

## What the script does

1. It programs the FPGA and starts the clock of the BRAM.
2. It writes a known word to each of the 1024 addresses: `0xA5A5A5` XOR the address. Then it reads all the words back with the correction off. If a word is wrong, the script stops and does not inject.
3. It injects `N_OF_INJECTIONS` upsets, one each 0.3 s, into random content bits of the BRAM (`TargetMemory.BRAM`). It stops the clock of the BRAM before each injection, and starts it again after the injection. The upsets add up, and the script does not correct them.
4. It reads all the words with the correction off, then with the correction on. Each time, it prints the wrong words.
5. It programs the FPGA again, to clear the upsets.

Set `N_OF_INJECTIONS` low to see the ECC correct each word. Set it higher to see bytes with two or more flipped bits, which the ECC cannot correct.

## Result

Sample output of a run on a Basys 3 (shortened). The frame addresses, the bit counts and the error counts depend on the board, the build and the seed.

```
Example 4: BRAM content injection and ECC recovery
SK-RAFT client 1.0.0, server 1.0.3
Programmed 210183BA2524A.
Writing known words in the BRAM...
Wrote 1024 words into pblock_soft_sec_bram.
Checking if all words round-trip...
All 1024 addresses read back correctly.
BRAM Content bit count = 36864
Press enter to stop injecting
Injecting BRAM Content bit upsets into pblock_soft_sec_bram, none corrected.
     #  FAR       WORD  BIT  BLK_TYPE
     1  00c00055    63   26    1
     2  00c00020    61   18    1
...
Injected 20 BRAM Content faults.
Reading the whole memory with ECC read off.
ECC off: 13 of 1024 addresses wrong.
  ADDR  EXPECTED                  READ                      FAULTY BITS
   104  101001011010010111001101  101001011010010110001101  6
   129  101001011010010100100100  101001111010010100100100  17
...
13 words wrong, 13 faulty bits in total.
--------------------------------------------------------------------------------
Reading the whole memory again with ECC read on.
ECC on: 0 of 1024 addresses wrong.
ECC on: every word read back correctly. Single flips were corrected; flips in check bits were masked.
Re-programming to clear the device.
Example 4 done.
```

How to read the output:

- **The upset lines** give the location of each injected bit: the frame address (FAR), the word in the frame and the bit in the word. `BLK_TYPE` is the block type of the frame: always 1, a BRAM content frame. `BRAM Content bit count` is 36864: 1024 words x 36 bits.
- **The wrong words** give the address (`ADDR`), the written word (`EXPECTED`), the word that the script read (`READ`) and the positions of the bits that differ (`FAULTY BITS`, 0 = the least significant bit).
- **Compare the two reads** to see what the ECC did with each word:

| Correction off | Correction on | Meaning |
| --- | --- | --- |
| Wrong | Correct | Each byte of the word had at most one flipped bit. The ECC corrected it. |
| Wrong | Wrong | A byte had two or more flipped bits. The ECC cannot correct them. |
| Correct | Wrong | The flips were in the check bits. The ECC changed a correct data bit. |
| Correct | Correct | No data bit is flipped. One flipped check bit does not show with the correction off, and the ECC hides it with the correction on. |

In the sample, the 13 wrong words each had one flipped bit, and the ECC corrected all of them.

## Background

### The configuration frames of the BRAM also affect it

The block type field of the frame address (FAR) tells the kind of a frame. The BRAM column has frames of two block types:

| Block type | Frames | What they hold |
| --- | --- | --- |
| 0 | Configuration (CLB, I/O, CLK) | The configuration of the BRAM, for example its width and its output register, and the interconnect of the column. |
| 1 | BRAM content | The data bits and the parity bits of the BRAMs. |

The script injects only into the block type 1 frames. An upset in the block type 0 frames of the column can also change the BRAM: it can change how the BRAM operates, stop it, or break a route to or from it. Thus, to test a design that uses BRAMs, also inject into the essential bits of the BRAM column (`TargetMemory.ESSENTIAL_CRAM` with `pblock=`), as [example 1](../example1/README.md) does for its pblocks.

AMD defines the block types in the description of the frame address register (FAR):

- UG470, *7 Series FPGAs Configuration User Guide*, section "Frame Address Register (FAR)".
- UG570, *UltraScale Architecture Configuration User Guide*, section "Frame Address Register (FAR)".

### The hold register

The design reads the BRAM through a hold register. `hw/create_project.tcl` adds it with the BRAM hold register pass ([`auxiliary_components/bram_hold_pass`](../../auxiliary_components/bram_hold_pass/README.md)), and the build prints the report of the pass. Without the hold register, each injection into the BRAM content would also change the BRAM output, and the design could read a wrong word after the clock starts again. Thus a wrong word in the results comes from the injected content, not from the test setup. [Example 6](../example6/README.md) explains why, and shows the effect on a BRAM without a hold register.

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device. |
| `Design(bit, ll, xdc)`, `dev.upload_design_to_server(design)`, `dev.program_fpga(design)` | Give the design files, upload them to the server and program the FPGA. |
| `dev.configure_seed(0)` | Set the seed of the random injections. The same seed gives the same upsets in each run. |
| `BscanPrimitive(user, length)`, `dev.write_bscan(primitive, value)`, `dev.read_bscan(primitive)` | Start and stop the clock (USER1), select the address (USER2), write and read the data (USER3), and enable the ECC correction (USER4). |
| `dev.bit_count(TargetMemory.BRAM, pblock=PBLOCK_BRAM)` | Get the number of content bits of the BRAM pblock. |
| `dev.inject_random(TargetMemory.BRAM, pblock=PBLOCK_BRAM)` | Inject one upset into a random content bit of the BRAM. |
| `dev.decode_far(far)` | Get the block type of the frame of each upset (1: BRAM content). |

## Run the example

Run the cells below in order. Set `SERIAL_NUMBER` and `BOARD` in the configuration cell. The design files must be in `hw/design_examples/<board>/brams_top/`: the first cells extract `design_examples.zip`.

On Linux

In [ ]:
%%bash
cd ../hw/design_examples/
./unzip_designs.sh

On Windows

In [ ]:
%%script pwsh
cd ..\hw\design_examples\
.\unzip_designs.bat

Install the toolchain

In [ ]:
%pip install ../../api/python

import IPython; IPython.get_ipython().kernel.do_shutdown(restart=True)

Set up the dependencies

In [ ]:
from skraft import BscanPrimitive, Design, SkraftError, SkraftFpga, TargetMemory

Start the server

In [ ]:
%%script bash --bg
sk-raft

Configure the values for the server connection

In [ ]:
SK_RAFT_SERVER_IP = "127.0.0.1"
SK_RAFT_SERVER_PORT = 50051

List all the devices and then pick the one you want with its serial number

In [ ]:
def main() -> int:
    try:
        # SK-RAFT: open the connection. Then list the connected boards.
        with SkraftFpga.connect(ip=SK_RAFT_SERVER_IP,
                                port=SK_RAFT_SERVER_PORT) as client:
            devices = client.get_devices()
    except SkraftError as e:
        print(f"Cannot list the boards: {e}")
        return 1
    except KeyboardInterrupt:
        print("Interrupted.")
        return 130

    if not devices:
        print("No board is connected to the SK-RAFT server.")
        return 1
    print(f"{'SERIAL NUMBER':16s} PART NUMBER")
    for device in devices:
        print(f"{device.serial_number:16s} {device.part_number}")
    return 0


if __name__ == "__main__":
    main()

Configure the values for the examples (especially `BOARD` and `SERIAL NUMBER`)

In [ ]:
# If you do not know the serial number of your target device,
# list the devices above.
SERIAL_NUMBER = "210183BA2524A"

# BOARD is the board of your target device: "basys3" or "zedboard". The
# notebook reads the design files from hw/design_examples/<BOARD>/brams_top/.
BOARD = "basys3"

Connect to the server

In [ ]:
client = SkraftFpga.connect(ip=SK_RAFT_SERVER_IP, port=SK_RAFT_SERVER_PORT)
dev = client.attach(SERIAL_NUMBER)

Import the time library

In [ ]:
import time

Change the design and configuration variables

In [ ]:
DESIGN_DIR = f"../hw/design_examples/{BOARD}/brams_top"

design = Design(
    bit=f"{DESIGN_DIR}/brams_top.bit",
    ll =f"{DESIGN_DIR}/brams_top.ll",
    xdc=f"{DESIGN_DIR}/board.xdc",
)

PBLOCK_BRAM = "pblock_soft_sec_bram"

# BSCAN registers: (USER number, width in bits)
USER_CLK_EN = BscanPrimitive(1, 1)
USER_ADDRESS = BscanPrimitive(2, 16)
USER_DATA = BscanPrimitive(3, 24)
USER_ECC_READ_EN = BscanPrimitive(4, 1)

# The design uses only the 10 low bits of the 16-bit USER2 register.
ADDRESS_BITS = 10
# The BRAM has 1024 words.
DEPTH = 1 << ADDRESS_BITS

DATA_MASK = (1 << USER_DATA.size_in_bits) - 1

Define more helper functions

In [ ]:
# The script prints one line for each injected bit. The line shows the
# location of the bit (FAR, word, bit). Then the line shows the block type
# from the decoded FAR (0 = configuration, 1 = BRAM content).
HIT_HEADER = "     #  FAR       WORD  BIT  BLK_TYPE"


def print_hit(dev, n: int, report) -> None:
    frame = dev.decode_far(report.far)
    print(f"{n:>6}  {report.far:08x}  {report.word_index:>4}  "
          f"{report.bit_index:>3}  {frame.blocktype!s:>3}  ")

# Start or stop the BRAM clock through USER1. The script stops the clock
# immediately before each injection below and starts it again immediately
# after. If the clock operates during an injection, the target circuit
# is active when the upset occurs. In a real campaign, the circuit can
# be unstable during the injection routine.
# Use this procedure as a general guideline for all campaigns.
def enable_clock(dev, enabled: bool) -> None:
    dev.write_bscan(USER_CLK_EN, 1 if enabled else 0)

# Select one BRAM address through USER2.
def select(dev, address: int) -> None:
    dev.write_bscan(USER_ADDRESS, address)


# Write the data word into USER3. write_bscan leaves Shift-DR through
# Exit1-DR -> Update-DR. For the design, this TAP path is a write to the
# selected address.
def write_word(dev, value: int) -> None:
    dev.write_bscan(USER_DATA, value)


# Read the data word from USER3. read_bscan leaves Shift-DR through
# Exit1-DR -> Pause-DR and does not go through Update-DR. Thus, the
# design does not write the bits that the read shifts in to the BRAM.
def read_word(dev) -> int:
    data = dev.read_bscan(USER_DATA)
    return int.from_bytes(data, "big") & DATA_MASK


# Start or stop ECC correction through USER4. The design always
# calculates and stores the check bits on each write. USER4 controls
# only the correction on the read path.
def ecc_read_en(dev, enabled: bool) -> None:
    dev.write_bscan(USER_ECC_READ_EN, 1 if enabled else 0)


# Write a known pattern to all words of the BRAM.
def fill_bram(dev) -> None:
    for address in range(DEPTH):
        select(dev, address)
        write_word(dev, (0xA5A5A5 ^ address) & DATA_MASK)
    print(f"Wrote {DEPTH} words into {PBLOCK_BRAM}.")


# Read all words of the BRAM. Return (address, expected, value) for each
# word that is different from the known pattern.
def check_bram(dev) -> list[tuple[int, int, int]]:
    corrupt = []
    # select() and read_word() are separate TAP scans. The second scan
    # cannot start before the first scan is done. The gap between the
    # scans is many TCK cycles. This gap is much longer than the BRAM read
    # latency plus the SEC decode time. Thus, USER3 always captures the
    # word at the new address.
    for address in range(DEPTH):
        select(dev, address)
        value = read_word(dev)
        expected = (0xA5A5A5 ^ address) & DATA_MASK
        if value != expected:
            corrupt.append((address, expected, value))
    return corrupt


# Print each wrong word as the expected bits and the read bits. Also print
# the bit positions (0 = LSB) that changed from the written value.
def print_wrong_words(words: list[tuple[int, int, int]]) -> None:
    if not words:
        return
    width = USER_DATA.size_in_bits
    print(f"{'ADDR':>6}  {'EXPECTED':<{width}}  {'READ':<{width}}  "
          f"FAULTY BITS")
    faulty_bits = 0
    for address, expected, value in words:
        flipped = [bit for bit in range(width)
                   if (expected ^ value) >> bit & 1]
        faulty_bits += len(flipped)
        print(f"{address:>6}  {expected:0{width}b}  {value:0{width}b}  "
              f"{', '.join(str(bit) for bit in flipped)}")
    print(f"{len(words)} words wrong, {faulty_bits} faulty bits in total.")

Program the board and upload the design to the server because it needs it for some operations.

In [ ]:
dev.upload_design_to_server(design)
dev.program_fpga(design)
print(f"programmed {SERIAL_NUMBER} with {design.bit}")

### Run the experiment!
The `N_OF_INJECTIONS` is the number of fault injections the experiment will do.

In [ ]:
# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)
enable_clock(dev, True)

# USER4 resets to 0 (correction off). The script writes 0 to
# USER4 anyway, so that the code shows the state.
# Read the raw data (no correction) for the round-trip check.
ecc_read_en(dev, False)
# Fill the memory array with known words.
print("Writing known words in the BRAM...")
fill_bram(dev)
# Make sure that each word reads back correctly before the
# injection starts.
print("Checking if all words round-trip...")
wrong = len(check_bram(dev))
if wrong:
    print(f"{wrong} of {DEPTH} addresses did not round-trip; not injecting anything.")
    raise SystemExit("Not all words round-trip.")
print(f"All {DEPTH} addresses read back correctly.")


In [ ]:
N_OF_INJECTIONS = 100

# Note: this count is the size of the set that inject_random
# selects from. The set contains all data bits and check
# bits of the BRAM (1K x 36 words).
# The upsets below accumulate. The script does not correct
# them.
bram_content_bits = dev.bit_count(TargetMemory.BRAM,
                                                  pblock=PBLOCK_BRAM)
print(f"BRAM Content bit count = {bram_content_bits}")
                
time.sleep(1)
print(f"Injecting BRAM Content bit upsets into {PBLOCK_BRAM}, none corrected.")
print(HIT_HEADER)
n = 0
iterator = N_OF_INJECTIONS
while iterator != 0:
    # Injection guideline: each content upset is a
    # readback-modify-writeback of a full frame of block
    # type 1. That frame holds bits of more than one BRAM in
    # the column. The server does not protect against user
    # traffic. Thus, keep all BRAMs in the target frames
    # idle (no reads or writes) during the injection.
    enable_clock(dev, False)
    report = dev.inject_random(TargetMemory.BRAM, 
                                               pblock=PBLOCK_BRAM)
    enable_clock(dev, True)
    print_hit(dev, n + 1, report)
    time.sleep(0.3)
    n += 1
    iterator -= 1

print(f"Injected {n} BRAM Content faults.")  
print("Reading the whole memory with ECC read off.")
# Scan the memory to find the wrong words.
err_words = check_bram(dev)
print(f"ECC off: {len(err_words)} of {DEPTH} addresses wrong.")
print_wrong_words(err_words)
print("-" * 80)
# Start the ECC correction on the read path.
ecc_read_en(dev, True)
print("Reading the whole memory again with ECC read on.")
# Scan the memory to find the wrong words.
corrupt_w_ecc = check_bram(dev)
print(f"ECC on: {len(corrupt_w_ecc)} of {DEPTH} addresses "
                      f"wrong.")
print_wrong_words(corrupt_w_ecc)

# The read path gives only the corrected data, with no error
# flag. With ECC off, flipped check bits do not show. With
# ECC on, SEC masks a single flipped check bit. SEC cannot
# detect two or more flipped bits in one byte. Then SEC can
# change a bit that is correct. This adds an error and
# does not remove one.
wrong_ecc_off = {address for address, _, _ in err_words}
wrong_ecc_on = {address for address, _, _ in corrupt_w_ecc}
if not wrong_ecc_on:
    print("ECC on: every word read back correctly. Single "
                          "flips were corrected; flips in check bits were "
                          "masked.")
else:
    print(f"ECC on: {len(wrong_ecc_on)} address(es) still wrong - "
                          f"at least one byte had two or more flips, which "
                          f"SEC can't correct.")
new = wrong_ecc_on - wrong_ecc_off
if new:
    print(f"{len(new)} of them read correctly with ECC "
                              f"off: the flips were in check bits and the "
                              f"decoder mis-corrected a data bit.")

In [ ]:
print("Re-programming to clear the device.")
dev.program_fpga(design)